# Mortgage Default — Feature Engineering (02)

This notebook builds customer-level balance features from the daily end-of-day balance dataset,
using only information prior to 2016. The result is a single modeling-ready dataset.


In [1]:
from pathlib import Path
import os
import sys

#check project root
current_path = Path.cwd()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
elif current_path.name == "mortgage-default-prediction":
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path

os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))

print("Working directory:", Path.cwd())



Working directory: /path/to/mortgage-default-prediction


In [2]:
import pandas as pd

from src.data_io import load_case_datasets
from src.feature_engineering import (
    build_balance_features_per_product,
    aggregate_product_features_to_customer,
)

pd.set_option("display.max_columns", 200)


In [3]:
datasets = load_case_datasets("data/case_data")

case_eod_balances = datasets.eod_balances
case_cust_prod_mapping = datasets.cust_prod_mapping
case_misc_features = datasets.misc_features
case_target = datasets.target


### Temporal Cutoff & Leakage Prevention

In [4]:
case_eod_balances["ymd"] = pd.to_datetime(case_eod_balances["ymd"])

print("Min date:", case_eod_balances["ymd"].min())
print("Max date:", case_eod_balances["ymd"].max())


Min date: 2013-01-01 00:00:00
Max date: 2016-01-01 00:00:00


In [5]:
cutoff_date = pd.Timestamp("2015-12-31")

case_eod_balances = case_eod_balances[
    case_eod_balances["ymd"] <= cutoff_date
].copy()

print("New max date:", case_eod_balances["ymd"].max())


New max date: 2015-12-31 00:00:00


**Temporal Validation**

Since the target is defined based on default behavior after January 1, 2016,
all balance-based features are constructed using data up to December 31, 2015.

This prevents look-ahead bias and ensures proper temporal separation between
feature construction and target definition.


### Create customer base

In [6]:
cust_base = case_misc_features.merge(case_target, on="cust_id", how="inner")
print("cust_base shape:", cust_base.shape)


cust_base shape: (5000, 5)


### Build product-level features

In [7]:
prod_features = build_balance_features_per_product(case_eod_balances)
print("prod_features shape:", prod_features.shape)
display(prod_features.head())


### Aggregate to customer-level

In [8]:
cust_balance_features = aggregate_product_features_to_customer(
    prod_features=prod_features,
    cust_prod_mapping=case_cust_prod_mapping,
)

print("cust_balance_features shape:", cust_balance_features.shape)
display(cust_balance_features.head())


#### Interpretation of Engineered Balance Features

The balance-based features are constructed in two steps:
1. Compute summary statistics at product level over time.
2. Aggregate those statistics to customer level across all products.

These variables aim to capture liquidity level, variability, and financial stress behavior.

**Balance Level & Volatility**

- **bal_mean_mean**  
  Average daily balance, averaged across all products of the customer.  
  Captures overall liquidity level.

- **bal_std_mean**  
  Average of the standard deviation of daily balances per product.  
  Measures balance volatility (stability vs fluctuation).

- **bal_min_mean**  
  Average of the minimum observed balance per product.  
  Indicates how deeply balances drop at their worst points.

- **bal_max_mean**  
  Average of the maximum observed balance per product.  
  Reflects peak liquidity levels.

**Negative-Balance Behavior (Financial Stress Indicators)**

- **neg_days_sum**  
  Total number of days with negative balance across all products.  
  Direct signal of financial stress.

- **neg_ratio_mean**  
  Average share of negative-balance days per product.  
  Normalized measure of how frequently the customer operates below zero.

- **n_days_sum**  
  Total observed days across all products.  
  Proxy for data coverage and product activity duration.

---

**Conclusion**

The engineered features summarize liquidity level, volatility, and frequency of financial stress across all products held by a customer. In particular, negative-balance frequency and depth-related indicators provide economically meaningful signals that are suitable for predictive modeling of default risk.


### Merge into modeling table

In [9]:
df_model = cust_base.merge(cust_balance_features, on="cust_id", how="left")

print("df_model shape:", df_model.shape)
display(df_model.head())


### Minimal missingness check

In [10]:
missing = df_model.isna().mean().sort_values(ascending=False)
print('Missingness check')
display(missing.head(15))


Missingness check


cust_segment      0.0588
age_group         0.0416
bal_mean_mean     0.0006
bal_std_mean      0.0006
bal_min_mean      0.0006
bal_max_mean      0.0006
neg_ratio_mean    0.0006
cust_id           0.0000
num_products      0.0000
target            0.0000
n_days_sum        0.0000
neg_days_sum      0.0000
dtype: float64

**conclusion**

Missingness is limited and concentrated in customer-level categorical features (cust_segment ~5.9% and age_group ~4.2%).

Balance-based features show negligible missingness (<0.1%), indicating that the aggregation process was successful and most customers have historical balance information.

Overall, missing values are manageable and can be addressed during modeling with simple imputation strategies.

### Save output

In [12]:
out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "df_model_v1.csv"
to_save = False
if to_save:
    df_model.to_csv(out_path, index=False)

print("Saved:", out_path)


Saved: data/processed/df_model_v1.csv


## Summary of feature engineering

Customer-level balance features were constructed by aggregating daily product balances into a small set of summary metrics.
The resulting dataset combines static customer attributes, default labels, and balance-based indicators, and is ready for modeling.
